In [2]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

date = datetime.now().strftime("%Y-%m-%d")

verbose = 2
with open("/home/chenzihao/workspace/cc2cc_test5/cc2cc/utils/gmtkn-def2.json") as f:
    json_data = json.load(f)

basis_args = "cc-pVTZ"
# basis_args = "def2-TZVPD"
print(basis_args)

dft_type_list = ["cc_ene", "b3lyp_ene", "b3lyp-d3bj_ene"]
data = pd.read_csv(
    f"/home/chenzihao/workspace/cc2cc_test5/validate_hkqai/ccdft_{basis_args}__gmtkn-def2.csv"
)

with open(f"./subset.json") as f:
    full_subset_dict = json.load(f)["full_subset_dict"]

for name_set, subset_list_ in full_subset_dict.items():
    full_subset_dict[name_set] = np.sort(subset_list_)
data_subset = {}

for name_set, subset_list_ in full_subset_dict.items():
    for dft_type in dft_type_list:
        if dft_type in data.columns:
            data_name = (data["name"].str.split(f"_{basis_args}").str[0]).to_numpy()
            data_dft = data[dft_type].to_numpy() * 627.5094733748099
        else:
            data_name = []
            for i_subset in subset_list_:
                if i_subset == "BH76RC":
                    data_name.append(json_data["molecule_BH76"])
                else:
                    data_name.append(json_data[f"molecule_{i_subset}"])
            data_name = np.concatenate(data_name)
            data_dft = np.zeros_like(data_name, dtype=float)

        for i_subset in subset_list_:
            name_subset = f"{dft_type}_{i_subset}"
            data_subset[name_subset] = {
                "name": [],
                "dft": [],
                "cc": [],
            }

            if i_subset == "BH76RC":
                molecular_list = json_data["molecule_BH76"]
            else:
                molecular_list = json_data[f"molecule_{i_subset}"]

            for i_molecule_name in molecular_list:
                col = np.where(data_name == i_molecule_name)[0]
                if col.size != 1:
                    if verbose > 0:
                        print(f"Warning: {i_molecule_name} not found in data file")
                    continue

            reaction_dict = json_data[f"reaction-{i_subset}"]
            for i_reaction_name, i_reaction in reaction_dict.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                finished, exist = True, True

                for i in range(len(systems_list)):
                    mole_name = (
                        systems_list[i]
                        if i_subset == "BH76RC"
                        else f"{i_subset}-{systems_list[i]}"
                    )
                    stoichiometry = int(stoichiometry_list[i])

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished, exist = False, False
                        if verbose > 0:
                            print(f"Warning: {mole_name} not found in json, ERROR")
                        break

                    col = np.where(data_name == mole_name)[0]
                    if col.size == 1:
                        atomic_energy_dft += data_dft[col[0]] * stoichiometry
                    else:
                        finished = False
                        if verbose > 0:
                            print(f"Warning: {mole_name} not found in data csv file")
                        break

                if exist:
                    data_subset[name_subset]["name"].append(i_reaction_name)
                if finished:
                    atomic_energy_cc = i_reaction["reference"]
                    data_subset[name_subset]["dft"].append(
                        abs(atomic_energy_dft - atomic_energy_cc)
                    )
                    data_subset[name_subset]["cc"].append(abs(atomic_energy_cc))
                    if np.abs(atomic_energy_cc) > 1000:
                        print(
                            f"Warning: {i_reaction_name} in {name_subset} has a large CC energy: {atomic_energy_cc} kcal/mol"
                        )

            for key, val in data_subset.items():
                for key2, val2 in val.items():
                    if isinstance(val2, list):
                        data_subset[key][key2] = np.array(val2)

            if verbose > 1:
                argsort_atomic_energy_dft = np.argsort(data_subset[name_subset]["dft"])[
                    ::-1
                ]
                for i in range(len(argsort_atomic_energy_dft)):
                    print(
                        f"Top {i+1} with name {data_subset[name_subset]["name"][argsort_atomic_energy_dft[i]]} DFT: {data_subset[name_subset]['dft'][argsort_atomic_energy_dft[i]]} kcal/mol"
                    )

header = dft_type_list + ["Processed"]

df_summary_subset = pd.DataFrame(columns=header)
mean_subset = pd.DataFrame(columns=header)
wtmad_1_subset = pd.DataFrame(columns=header)
wtmad_2_subset = pd.DataFrame(columns=header)

for dft_type in dft_type_list:
    mean_absolute_deviation_list = []

    for name_set, subset_list_ in full_subset_dict.items():
        subset_dft = []
        wtmad_1_dft = []
        wtmad_2_dft = []
        processed = []

        for i_subset in subset_list_:
            name_subset = f"{dft_type}_{i_subset}"

            if len(data_subset[name_subset]["dft"]) == 0:
                df_summary_subset.loc[i_subset, dft_type] = 0
                df_summary_subset.loc[i_subset, "Processed"] = (
                    f"0 / {len(data_subset[name_subset]['name'])}"
                )
            else:
                df_summary_subset.loc[i_subset, dft_type] = np.mean(
                    data_subset[name_subset]["dft"]
                )
                df_summary_subset.loc[i_subset, "Processed"] = (
                    "DONE"
                    if (
                        (
                            len(data_subset[name_subset]["dft"])
                            == len(data_subset[name_subset]["name"])
                        )
                        and (len(data_subset[name_subset]["dft"]) != 0)
                    )
                    else f"{len(data_subset[name_subset]['dft'])} / "
                    f"{len(data_subset[name_subset]['name'])}"
                )

                if np.mean(data_subset[name_subset]["cc"]) > 75:
                    wtmad_1 = 0.1
                elif np.mean(data_subset[name_subset]["cc"]) < 7.5:
                    wtmad_1 = 10
                else:
                    wtmad_1 = 1

                subset_dft = np.append(subset_dft, data_subset[name_subset]["dft"])
                wtmad_1_dft = np.append(
                    wtmad_1_dft,
                    wtmad_1 * np.mean(data_subset[name_subset]["dft"]),
                )
                wtmad_2_dft = np.append(
                    wtmad_2_dft,
                    data_subset[name_subset]["dft"]
                    / np.mean(data_subset[name_subset]["cc"]),
                )
                mean_absolute_deviation_list = np.append(
                    mean_absolute_deviation_list,
                    data_subset[name_subset]["cc"],
                )
            if (
                len(data_subset[name_subset]["dft"])
                == len(data_subset[name_subset]["name"])
                and len(data_subset[name_subset]["name"]) > 0
            ):
                processed.append(1)
            else:
                processed.append(0)

        mean_subset.loc[name_set, dft_type] = np.mean(subset_dft)
        wtmad_1_subset.loc[name_set, dft_type] = np.mean(wtmad_1_dft)
        wtmad_2_subset.loc[name_set, dft_type] = np.sum(wtmad_2_dft)
        mean_subset.loc[name_set, "Processed"] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_1_subset.loc[name_set, "Processed"] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )
        wtmad_2_subset.loc[name_set, "Processed"] = (
            "DONE"
            if (sum(processed) == len(processed) and len(processed) != 0)
            else f"{sum(processed)} / " f"{len(processed)}"
        )

    mean_absolute_deviation = 56.84 / len(
        mean_absolute_deviation_list
    )
    print(
        f"Mean absolute deviation for {dft_type}: {mean_absolute_deviation:.4f} kcal/mol"
    )
    for name_set in full_subset_dict.keys():
        wtmad_2_subset.loc[name_set, dft_type] = (
            mean_absolute_deviation * wtmad_2_subset.loc[name_set, dft_type]
        )

    wtmad_1_subset.loc["summary", "Processed"] = "--"
    wtmad_2_subset.loc["summary", "Processed"] = "--"
    wtmad_1_subset.loc["summary", dft_type] = 0
    wtmad_2_subset.loc["summary", dft_type] = 0
    for name_set in full_subset_dict.keys():
        wtmad_1_subset.loc["summary", dft_type] += wtmad_1_subset.loc[
            name_set, dft_type
        ]
        wtmad_2_subset.loc["summary", dft_type] += wtmad_2_subset.loc[
            name_set, dft_type
        ]

print("MAE")
display(mean_subset)
print("wtmad_1")
display(wtmad_1_subset)
print("wtmad_2")
display(wtmad_2_subset)
print("Summary of Subset")
print("MAE")
display(df_summary_subset)

# # save summary to csv with date
# df_summary_subset_ele.to_csv(f"../validate_hkqai_done/df_summary_subset_ele_{date}.csv")
# df_summary_subset.to_csv(f"../validate_hkqai_done/summary_subset_{date}.csv")
# mean_subset.to_csv(f"../validate_hkqai_done/mean_subset_{date}.csv")
# wtmad_1_subset.to_csv(f"../validate_hkqai_done/wtmad_1_subset_{date}.csv")
# wtmad_2_subset.to_csv(f"../validate_hkqai_done/wtmad_2_subset_{date}.csv")
# # save summary to excel with date
# df_summary_subset_ele.to_excel(
#     f"../validate_hkqai_done/df_summary_subset_ele_{date}.xlsx"
# )
# df_summary_subset.to_excel(f"../validate_hkqai_done/summary_subset_{date}.xlsx")
# mean_subset.to_excel(f"../validate_hkqai_done/mean_subset_{date}.xlsx")
# wtmad_1_subset.to_excel(f"../validate_hkqai_done/wtmad_1_subset_{date}.xlsx")
# wtmad_2_subset.to_excel(f"../validate_hkqai_done/wtmad_2_subset_{date}.xlsx")

cc-pVTZ
Top 1 with name 1 DFT: 51.6 kcal/mol
Top 2 with name 0 DFT: 38.5 kcal/mol
Top 3 with name 3 DFT: 38.4 kcal/mol
Top 4 with name 2 DFT: 32.5 kcal/mol
Top 5 with name 4 DFT: 31.2 kcal/mol
Top 6 with name 5 DFT: 23.1 kcal/mol
Top 1 with name 2 DFT: 131.13 kcal/mol
Top 2 with name 0 DFT: 86.47 kcal/mol
Top 3 with name 5 DFT: 66.28 kcal/mol
Top 4 with name 6 DFT: 56.55 kcal/mol
Top 5 with name 1 DFT: 53.15 kcal/mol
Top 6 with name 4 DFT: 47.42 kcal/mol
Top 7 with name 3 DFT: 34.51 kcal/mol
Top 8 with name 7 DFT: 25.3 kcal/mol
Top 1 with name 3 DFT: 142.1 kcal/mol
Top 2 with name 5 DFT: 139.2 kcal/mol
Top 3 with name 0 DFT: 138.7 kcal/mol
Top 4 with name 4 DFT: 117.5 kcal/mol
Top 5 with name 1 DFT: 106.6 kcal/mol
Top 6 with name 2 DFT: 96.2 kcal/mol
Top 7 with name 6 DFT: 82.5 kcal/mol
Top 8 with name 9 DFT: 65.2 kcal/mol
Top 9 with name 7 DFT: 62.2 kcal/mol
Top 10 with name 8 DFT: 56.7 kcal/mol
Top 1 with name BH76RC_3 DFT: 103.28 kcal/mol
Top 2 with name BH76RC_1 DFT: 64.91 kcal/mol

/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/fromnumeric.py:3504: RuntimeWarning: Mean of empty slice.
  return _methods._mean(a, axis=axis, dtype=dtype,
/home/chenzihao/anaconda3/envs/pyscf/lib/python3.12/site-packages/numpy/core/_methods.py:129: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


,cc_ene,b3lyp_ene,b3lyp-d3bj_ene,Processed
sub1,155.762918,5.03683,4.627682,3 / 18
sub2,97.057338,36.89058,15.174867,1 / 9
sub3,22.721031,NaN,NaN,0 / 7
sub4,18.410852,1.385127,1.236351,0 / 12
sub5,3.557344,0.741932,0.290605,0 / 9


wtmad_1


,cc_ene,b3lyp_ene,b3lyp-d3bj_ene,Processed
sub1,37.583104,2.84512,2.828378,3 / 18
sub2,26.253228,7.972577,2.910386,1 / 9
sub3,35.39219,NaN,NaN,0 / 7
sub4,33.0659,13.851267,12.36351,0 / 12
sub5,31.862853,9.556913,3.221594,0 / 9
summary,164.157275,NaN,NaN,--


wtmad_2


,cc_ene,b3lyp_ene,b3lyp-d3bj_ene,Processed
sub1,17.864,1.620095,1.57518,3 / 18
sub2,9.177488,4.50525,1.57478,1 / 9
sub3,7.326884,0.0,0.0,0 / 7
sub4,11.481302,1.406328,1.255275,0 / 12
sub5,10.990326,1.210652,0.499207,0 / 9
summary,56.84,8.742324,4.904443,--


Summary of Subset
MAE


,cc_ene,b3lyp_ene,b3lyp-d3bj_ene,Processed
AL2X6,35.883333,0,0,0 / 6
ALK8,62.60125,0,0,0 / 8
ALKBDE10,100.69,0,0,0 / 10
BH76RC,21.391667,0,0,0 / 30
DC13,54.978462,0,0,0 / 13
DIPCS10,654.26,3.794038,3.781285,9 / 10
FH51,31.01098,0,0,0 / 51
G21EA,33.624,10.168724,10.163771,DONE
G21IP,257.609583,3.806944,3.813164,DONE
G2RC,51.2632,0,0,0 / 25


In [2]:
mean_absolute_deviation_list

[]